In [1]:
import infercnvpy as cnv
import scanpy as sc
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
import os

In [ ]:
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, to_tree
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib import gridspec
from scipy.cluster.hierarchy import dendrogram

In [ ]:
sample_list_st = [
  "P1","P2","P3","P4","P5","P6","P7","P8","P9","P10","P11","P12","P13","P14","P15","P16","P26","P27","P30","P32","P33","P34","P35","P36","P37","P38","P39",
  "P40_1","P40_2","P42","P43_1","P43_2","P44","P45","P46","P47","P48","P49"
]

In [ ]:
for sample_id in sample_list_st[19:len(sample_list_st)]:

    output_dir=f'E:/Projects/BRCA/results/ST/infercnv_metacell/{sample_id}'

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    sc.settings.figdir=output_dir
    mask_sample = (
        (adata.obs["sample"] == sample_id)
        & (adata.obs["celltype"] == "Epithelial")
        & (adata.obs["cnv_score"] >= 0)
    )
    mask_gene = adata.var["chromosome"].notna().values
    ad = adata[mask_sample, mask_gene].copy()  
    cnv.pl.chromosome_heatmap(ad, groupby="celltype")
    Xcnv = ad.obsm["X_cnv"].copy()         # cells × genes
    var  = ad.var.reset_index(drop=True)  
    # 1) Denoise: map values in [0.98, 1.02] to 1
    # infercnvpy returns logFC values, which may be negative; skip this step
    X = Xcnv.toarray().copy()

    thr = 0.03      # Corresponds to 0.98-1.02
    X[(X >= -thr) & (X <= thr)] = 0.0   # Flatten small fluctuations to 0 (neutral)
    ad.obsm["X_cnv"] = X
    cnv.pl.chromosome_heatmap(ad, groupby="celltype",save='0.3.heatmap_denoised.png')

    # ===== 2) Use uns['cnv']['chr_pos'] to obtain bin boundaries for each chromosome and aggregate by segment =====
    assert "cnv" in ad.uns and "chr_pos" in ad.uns["cnv"], "Missing ad.uns['cnv']['chr_pos']"
    import re
    pos = ad.uns["cnv"]["chr_pos"]  # dict: {'chr1':0, 'chr2':157, ...}
    def chr_key(c):
        m = re.match(r"^chr(\d+)$", c)
        if m: return (0, int(m.group(1)))
        if c == "chrX": return (1, 23)
        if c == "chrY": return (1, 24)
        return (2, c)

    chroms = sorted(pos.keys(), key=chr_key)
    import numpy as np
    bounds = []
    for i, ch in enumerate(chroms):
        s = int(pos[ch])
        e = int(pos[chroms[i+1]]) if i+1 < len(chroms) else X .shape[1]
        bounds.append((ch, s, e))
    # 2) Assemble autosomal column indices and bin names for recognition in R
    keep_idx = np.concatenate([np.arange(s, e) for ch, s, e in bounds]).astype(int)
    bin_names = []
    for ch, s, e in bounds:
        for k, j in enumerate(range(s, e), start=1):
            bin_names.append(f"{ch}_bin{k:04d}")

    # 3) Extract the autosomal matrix (rows=spots, columns=bins) and optionally clip display values to match R
    X_auto = X[:, keep_idx].copy()
    # Optional: use a consistent color range by clipping values outside 1 +/- fig_thres
    obs_df = pd.DataFrame(X_auto.T, index=bin_names, columns=ad.obs_names.astype(str))
    obs_df.to_csv(f"{output_dir}/0.3.infercnv.observations.txt", sep=" ", header=True, index=True)
    gene_order = pd.DataFrame({"V1": bin_names, "V2": [ch for ch,s,e in bounds for _ in range(e-s)]})
    gene_order.to_csv(f"{output_dir}/0.3.GeneOrderFile.txt", sep="\t", header=False, index=False)

    # Export the reference matrix; cluster 6 is a relatively pure T-cell population
    mask_sample = ( 
        (adata.obs["celltype"] == "T")
        & (adata.obs["cnv_leiden"] == '0')
    )
    print("Number of True values:", mask_sample.sum())
    mask_gene = adata.var["chromosome"].notna().values
    ad_ref = adata[mask_sample, mask_gene].copy()             # AnnData for a single sample
    cnv.pl.chromosome_heatmap(
        ad_ref,
        groupby="celltype",
        vmin=-0.3,   # Alternatively use -0.3 / -0.15; adjust to the scale of the data
        vmax= 0.3
    )
    cnv.pl.umap(
        ad_ref,
        color="cnv_leiden",
        legend_loc="on data",
        legend_fontoutline=2,
        show=True,
    )
    X_ref = ad_ref.obsm["X_cnv"].copy()         # cells × genes
    X_ref= X_ref.toarray().copy()
    thr = 0.03      # Corresponds to 0.98-1.02
    X_ref[(X_ref >= -thr) & (X_ref <= thr)] = 0.0   # Flatten small fluctuations to 0 (neutral)
    ad_ref.obsm["X_cnv"] = X_ref
    cnv.pl.chromosome_heatmap(
        ad_ref,
        groupby="celltype",
        vmin=-0.3,   # Alternatively use -0.3 / -0.15; adjust to the scale of the data
        vmax= 0.3
    )
    ref_df = pd.DataFrame(X_ref.T, index=bin_names, columns=ad_ref.obs_names.astype(str))
    ref_df.to_csv(f"{output_dir}/0.3.infercnv.reference.txt", sep=" ", header=True, index=True)

    # === 2) Perform hierarchical clustering on the bin-level matrix without aggregating chromosome means ===
    from scipy.spatial.distance import pdist
    from scipy.cluster.hierarchy import linkage, to_tree
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
    from matplotlib import gridspec
    from scipy.cluster.hierarchy import dendrogram
    D = pdist(X, metric="Euclidean")
    Z = linkage(D, method="ward")

    # 1) Obtain the leaf order (cell order) from Z
    # bounds: [(chr, s, e), ...]; s/e are half-open column index intervals
    chr_edges = [e for (_, _, e) in bounds]                 # Right boundaries for vertical separator lines
    chr_mids  = [ (s + e) // 2 for (_, s, e) in bounds ]    # Midpoints for labels
    chr_names = [ ch.replace("chr","") for (ch, _, _) in bounds ]

    # 1) Stretch heights with a concave function: extend the root and compress leaf tips
    alpha = 1.2          # 0.3-0.6: smaller values compress the root more
    Z_mod = Z.copy()
    d = Z_mod[:, 2]
    dmax = d.max()
    Z_mod[:, 2] = (d / dmax) ** alpha * dmax   # Keep the maximum height unchanged

    fig = plt.figure(figsize=(10, 8))
    gs  = gridspec.GridSpec(ncols=3, nrows=1, width_ratios=[5, 0.15, 8], wspace=0.05)

    ax_tree = fig.add_subplot(gs[0,0])
    # Plot the full tree without truncation
    ddata = dendrogram(
        Z_mod,
        orientation="left",
        no_labels=True,
        color_threshold=None,
        ax=ax_tree,
        above_threshold_color="k"
    )

    ax_tree.invert_yaxis()
    ax_tree.axis("off")
    leaf_order = ddata["leaves"]                  # Indices based on the row order of X
    X_plot = X[leaf_order, :]           
    # Change the width of all lines
    for line in ax_tree.collections:
        line.set_linewidth(0.5)   # Use thinner lines


    # Optionally label upstream nodes, for example nodes 0 through 10
    # Reorder cells according to the tree
    cmap = LinearSegmentedColormap.from_list("cnv_cmap", ["navy", "white", "firebrick"])
    # Set lower and upper thresholds dynamically to limit the effect of extremes; alternatively fix vmin=0.8, vmax=1.2
    vmin = np.percentile(X_plot, 1)
    vmax = np.percentile(X_plot, 99)
    # Center symmetrically at 0
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
    # 4b) Heatmap on the right
    ax_hm = fig.add_subplot(gs[0,2])
    im = ax_hm.imshow(X_plot, aspect="auto", interpolation="nearest", cmap=cmap, norm=norm)
    ax_hm.set_yticks([])             # Hide crowded cell names; use a small number of ticks if needed
    ax_hm.set_xticks(chr_mids)
    ax_hm.set_xticklabels(chr_names, rotation=0, fontsize=8)

    # Draw chromosome separators
    for edge in chr_edges:
        ax_hm.axvline(edge - 0.5, color="k", linewidth=0.5, alpha=0.5)

    ax_cb = fig.add_subplot(gs[0,1])
    cb = plt.colorbar(im, cax=ax_cb)
    out_png = f"{output_dir}/0.3.heatmap_with_dendrogram.png"
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    def _build_newick(node, parent_dist, leaf_names):
        if node.is_leaf():
            return f"{leaf_names[node.id]}:{(parent_dist - node.dist):.6f}"
        left = _build_newick(node.left, node.dist, leaf_names)
        right = _build_newick(node.right, node.dist, leaf_names)
        return f"({left},{right}):{(parent_dist - node.dist):.6f}"

    root, _ = to_tree(Z_mod, rd=True)
    leaf_names = ad.obs_names.tolist()        # Keep the same row order as X
    newick_str = _build_newick(root, root.dist, leaf_names) + ";\n"

    # === 4) Write files using inferCNV-compatible filenames ===
    out_path = f"{output_dir}/0.3.obs_dendro_Zmod.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(newick_str)

    print("Newick dendrogram (bin-level) written to:", out_path)

    # Convert numpy.ndarray to pandas DataFrame
    X_df = pd.DataFrame(X_plot)
    X_outpath = f"{output_dir}/0.3.X_plot.txt"
    X_df.to_csv(X_outpath, sep="\t", header=True, index=True)